In [1]:
from pathlib import Path
import pandas as pd

# Define dataset path
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

HASHTAGS_PATH = DATA_DIR / "hashtags.json"

if not HASHTAGS_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {HASHTAGS_PATH}"
    )

# Load original dataset
df_hashtags_raw = pd.read_json(HASHTAGS_PATH)

# Create working copy
df_hashtags_clean = df_hashtags_raw.copy()

print("Original dataset shape:", df_hashtags_raw.shape)

print("\nOriginal data types:")
display(df_hashtags_raw.dtypes.to_frame("dtype"))

print("\nDataset preview:")
display(df_hashtags_raw.head())

Original dataset shape: (205214, 5)

Original data types:


,dtype
id_hashtag,int64
first_seen,str
hashtag,str
truth_id,int64
hashtag_id,int64



Dataset preview:


,id_hashtag,first_seen,hashtag,truth_id,hashtag_id
0,1,2022-09-14 17:29:14.578577,MarkGeist,18,1
1,2,2022-09-14 17:34:04.180279,LIVGolf,21,2
2,2,2022-09-14 17:34:04.180279,LIVGolf,478624,2
3,3,2022-09-14 23:15:11.267669,TrumpTurnberry,37,3
4,3,2022-09-14 23:15:11.267669,TrumpTurnberry,384189,3


In [2]:
# Count exact duplicate records
exact_duplicates = df_hashtags_clean.duplicated().sum()

# Count duplicated hashtag-post relationships
duplicate_pairs = df_hashtags_clean.duplicated(
    subset=["id_hashtag", "truth_id"]
).sum()

print("DUPLICATE VALIDATION")
print("-" * 40)

print("Exact duplicate records:", exact_duplicates)
print("Duplicate hashtag-post pairs:", duplicate_pairs)

# Inspect whether pair duplicates contain conflicting information
conflicting_pair_duplicates = (
    duplicate_pairs - exact_duplicates
)

print(
    "Additional pair duplicates:",
    conflicting_pair_duplicates
)

# Remove exact duplicates only
df_hashtags_clean = (
    df_hashtags_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

# Final validation
remaining_duplicates = (
    df_hashtags_clean.duplicated().sum()
)

print("\nOriginal records:", len(df_hashtags_raw))
print("Duplicates removed:", exact_duplicates)
print("Cleaned records:", len(df_hashtags_clean))
print("Remaining exact duplicates:", remaining_duplicates)

assert remaining_duplicates == 0
assert len(df_hashtags_clean) == (
    len(df_hashtags_raw) - exact_duplicates
)

DUPLICATE VALIDATION
----------------------------------------
Exact duplicate records: 2
Duplicate hashtag-post pairs: 2
Additional pair duplicates: 0

Original records: 205214
Duplicates removed: 2
Cleaned records: 205212
Remaining exact duplicates: 0


In [3]:
# Define identifier columns
id_columns = [
    "id_hashtag",
    "hashtag_id",
    "truth_id"
]

# Convert identifiers to nullable integers
for col in id_columns:
    df_hashtags_clean[col] = pd.to_numeric(
        df_hashtags_clean[col],
        errors="coerce"
    ).astype("Int64")

# Check missing identifiers
missing_ids = df_hashtags_clean[id_columns].isna().sum()

# Check non-positive identifiers
non_positive_ids = (
    df_hashtags_clean[id_columns]
    .le(0)
    .sum()
)

# Verify consistency between hashtag identifiers
id_mismatches = (
    df_hashtags_clean["id_hashtag"]
    != df_hashtags_clean["hashtag_id"]
).fillna(False).sum()

# Convert first_seen to datetime
df_hashtags_clean["first_seen"] = pd.to_datetime(
    df_hashtags_clean["first_seen"],
    errors="coerce",
    utc=True
)

# Validate timestamps
invalid_timestamps = (
    df_hashtags_clean["first_seen"].isna().sum()
)

future_timestamps = (
    df_hashtags_clean["first_seen"]
    > pd.Timestamp.now(tz="UTC")
).sum()

# Build validation summary
identifier_validation = pd.DataFrame({
    "Check": [
        "Missing identifiers",
        "Non-positive identifiers",
        "Hashtag ID mismatches",
        "Missing or invalid timestamps",
        "Future timestamps"
    ],
    "Count": [
        missing_ids.sum(),
        non_positive_ids.sum(),
        id_mismatches,
        invalid_timestamps,
        future_timestamps
    ]
})

print("IDENTIFIER AND TIMESTAMP VALIDATION")
print("-" * 45)

display(identifier_validation)

print("\nData types after conversion:")
display(df_hashtags_clean.dtypes.to_frame("dtype"))

print("\nDataset shape:")
print(df_hashtags_clean.shape)

IDENTIFIER AND TIMESTAMP VALIDATION
---------------------------------------------


,Check,Count
0,Missing identifiers,0
1,Non-positive identifiers,0
2,Hashtag ID mismatches,0
3,Missing or invalid timestamps,0
4,Future timestamps,0



Data types after conversion:


,dtype
id_hashtag,Int64
first_seen,"datetime64[us, UTC]"
hashtag,str
truth_id,Int64
hashtag_id,Int64



Dataset shape:
(205212, 5)


In [4]:
# Preserve original hashtag values for validation
hashtag_values = df_hashtags_clean["hashtag"].astype("string")

# Identify missing and empty values
missing_hashtags = hashtag_values.isna().sum()

empty_hashtags = (
    hashtag_values.str.strip().eq("").fillna(False)
).sum()

# Identify leading or trailing whitespace
whitespace_issues = (
    hashtag_values.ne(hashtag_values.str.strip())
    .fillna(False)
).sum()

# Identify placeholder values
placeholder_values = {
    "nan", "none", "null", "<na>", "-1"
}

placeholder_mask = (
    hashtag_values.str.strip()
    .str.lower()
    .isin(placeholder_values)
)

# Identify whitespace inside hashtag names
internal_whitespace_mask = (
    hashtag_values.str.contains(r"\s", regex=True, na=False)
)

# Identify control characters
control_character_mask = (
    hashtag_values.str.contains(
        r"[\x00-\x1f\x7f]",
        regex=True,
        na=False
    )
)

# Build validation summary
hashtag_text_validation = pd.DataFrame({
    "Check": [
        "Missing hashtag names",
        "Empty hashtag names",
        "Leading/trailing whitespace",
        "Placeholder values",
        "Hashtags containing whitespace",
        "Control characters"
    ],
    "Count": [
        missing_hashtags,
        empty_hashtags,
        whitespace_issues,
        placeholder_mask.sum(),
        internal_whitespace_mask.sum(),
        control_character_mask.sum()
    ]
})

print("HASHTAG TEXT VALIDATION")
print("-" * 45)

display(hashtag_text_validation)

# Inspect suspicious records
suspicious_mask = (
    hashtag_values.isna()
    | hashtag_values.str.strip().eq("").fillna(False)
    | placeholder_mask
    | internal_whitespace_mask
    | control_character_mask
)

print("\nExamples of potentially problematic hashtags:")

display(
    df_hashtags_clean.loc[
        suspicious_mask,
        ["id_hashtag", "hashtag", "truth_id"]
    ].drop_duplicates().head(20)
)

HASHTAG TEXT VALIDATION
---------------------------------------------


,Check,Count
0,Missing hashtag names,0
1,Empty hashtag names,0
2,Leading/trailing whitespace,0
3,Placeholder values,0
4,Hashtags containing whitespace,1
5,Control characters,0



Examples of potentially problematic hashtags:


,id_hashtag,hashtag,truth_id
189771,20299,"LegalImmigrants' rel='""nofollow noopener'"" tar...",718216


In [5]:
# Identify the suspicious hashtag
malformed_hashtag_mask = (
    df_hashtags_clean["hashtag"]
    .astype("string")
    .str.contains(
        r'rel\s*=\s*["\']|<[^>]+>',
        regex=True,
        na=False
    )
)

# Preserve suspicious records
df_malformed_hashtags = df_hashtags_clean.loc[
    malformed_hashtag_mask
].copy()

print("MALFORMED HASHTAG INVESTIGATION")
print("-" * 45)

print("Malformed records:",
      len(df_malformed_hashtags))

print("Affected hashtag IDs:",
      df_malformed_hashtags["id_hashtag"].nunique())

print("\nFull malformed values:")

for value in df_malformed_hashtags["hashtag"].unique():
    print(repr(value))

# Check all associations involving affected hashtag IDs
affected_ids = df_malformed_hashtags[
    "id_hashtag"
].unique()

related_records = df_hashtags_clean.loc[
    df_hashtags_clean["id_hashtag"].isin(
        affected_ids
    )
]

print("\nAll records with affected hashtag IDs:")
display(related_records)

MALFORMED HASHTAG INVESTIGATION
---------------------------------------------
Malformed records: 1
Affected hashtag IDs: 1

Full malformed values:
'LegalImmigrants\' rel=\'"nofollow noopener\'" target=\'"_blank\'" class=\'"status-link\'" title=\'"https://truthsocial.com/tags/LegalImmigrants\'"><span class=\'"invisible\'">https://</span><span class=\'"ellipsis\'">truthsocial.com/tags/LegalImmi</span><span class=\'"invisible\'">grants</span></a> Are Awesome As In <a href=\'"https://truthsocial.com/tags/ElonMusk"'

All records with affected hashtag IDs:


,id_hashtag,first_seen,hashtag,truth_id,hashtag_id
189771,20299,2022-11-04 02:55:23.192836+00:00,"LegalImmigrants' rel='""nofollow noopener'"" tar...",718216,20299


In [6]:
# Identify the malformed hashtag record
malformed_mask = df_hashtags_clean["id_hashtag"].eq(20299)

# Preserve original values
df_hashtag_recovery = df_hashtags_clean.loc[
    malformed_mask,
    ["id_hashtag", "hashtag", "truth_id"]
].copy()

# Extract hashtag from embedded Truth Social URL
url_candidate = (
    df_hashtag_recovery["hashtag"]
    .astype("string")
    .str.extract(
        r"truthsocial\.com/tags/([A-Za-z0-9_]+)",
        expand=False
    )
)

# Extract hashtag prefix before HTML contamination
prefix_candidate = (
    df_hashtag_recovery["hashtag"]
    .astype("string")
    .str.extract(
        r"^([A-Za-z0-9_]+)['\"]?\s+rel=",
        expand=False
    )
)

# Validate agreement between extracted values
valid_recovery = (
    url_candidate.notna()
    & prefix_candidate.notna()
    & url_candidate.eq(prefix_candidate)
)

print("HASHTAG RECOVERY VALIDATION")
print("-" * 45)

print("Malformed records:", len(df_hashtag_recovery))
print("Validated recoveries:", valid_recovery.sum())

print("\nRecovery candidates:")

display(
    pd.DataFrame({
        "id_hashtag": df_hashtag_recovery["id_hashtag"],
        "prefix_candidate": prefix_candidate,
        "url_candidate": url_candidate,
        "valid_recovery": valid_recovery
    })
)

# Apply correction only if validation succeeds
if len(df_hashtag_recovery) == 1 and valid_recovery.all():

    recovered_hashtag = url_candidate.iloc[0]

    df_hashtags_clean.loc[
        malformed_mask,
        "hashtag"
    ] = recovered_hashtag

    print("\nHashtag successfully recovered:")
    print(recovered_hashtag)

else:
    print("\nRecovery not applied: validation failed.")

# Inspect final record
print("\nFinal record:")

display(
    df_hashtags_clean.loc[
        malformed_mask,
        ["id_hashtag", "hashtag", "truth_id"]
    ]
)

HASHTAG RECOVERY VALIDATION
---------------------------------------------
Malformed records: 1
Validated recoveries: 1

Recovery candidates:


,id_hashtag,prefix_candidate,url_candidate,valid_recovery
189771,20299,LegalImmigrants,LegalImmigrants,True



Hashtag successfully recovered:
LegalImmigrants

Final record:


,id_hashtag,hashtag,truth_id
189771,20299,LegalImmigrants,718216


In [7]:
# Count distinct hashtag names per identifier
names_per_id = (
    df_hashtags_clean
    .groupby("id_hashtag")["hashtag"]
    .nunique()
)

# Identify IDs associated with multiple names
inconsistent_ids = names_per_id[
    names_per_id > 1
]

# Check consistency between redundant identifiers
id_mismatches = (
    df_hashtags_clean["id_hashtag"]
    != df_hashtags_clean["hashtag_id"]
).fillna(False).sum()

# Verify duplicate hashtag-post relationships
duplicate_pairs = df_hashtags_clean.duplicated(
    subset=["id_hashtag", "truth_id"]
).sum()

print("HASHTAG MAPPING CONSISTENCY")
print("-" * 45)

print("Unique hashtag IDs:",
      df_hashtags_clean["id_hashtag"].nunique())

print("IDs associated with multiple names:",
      len(inconsistent_ids))

print("Hashtag identifier mismatches:",
      id_mismatches)

print("Duplicate hashtag-post relationships:",
      duplicate_pairs)

print("\nInconsistent identifiers:")
display(inconsistent_ids.head(20))

# Validate recovered hashtag
recovered_record = df_hashtags_clean.loc[
    df_hashtags_clean["id_hashtag"] == 20299,
    "hashtag"
]

assert recovered_record.eq("LegalImmigrants").all()

HASHTAG MAPPING CONSISTENCY
---------------------------------------------
Unique hashtag IDs: 21599
IDs associated with multiple names: 0
Hashtag identifier mismatches: 0
Duplicate hashtag-post relationships: 0

Inconsistent identifiers:


Series([], Name: hashtag, dtype: int64)

In [8]:
# Define path to cleaned Truths dataset
TRUTHS_CLEAN_PATH = DATA_DIR / "truths_clean.csv"

if not TRUTHS_CLEAN_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {TRUTHS_CLEAN_PATH}"
    )

# Load post identifiers
df_truths_ids = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id"],
    dtype={"id": "Int64"}
)

# Extract unique post identifiers
valid_truth_ids = pd.Index(
    df_truths_ids["id"].dropna().unique()
)

# Check referential integrity
truth_exists = df_hashtags_clean["truth_id"].isin(
    valid_truth_ids
)

matched_relationships = truth_exists.sum()
unmatched_relationships = (~truth_exists).sum()

# Count distinct unmatched posts
unmatched_post_ids = (
    df_hashtags_clean.loc[
        ~truth_exists, "truth_id"
    ].nunique()
)

# Build summary
referential_summary = pd.DataFrame({
    "Check": [
        "Total hashtag-post relationships",
        "Matched relationships",
        "Unmatched relationships",
        "Distinct unmatched post IDs"
    ],
    "Count": [
        len(df_hashtags_clean),
        matched_relationships,
        unmatched_relationships,
        unmatched_post_ids
    ]
})

print("REFERENTIAL INTEGRITY — MAIN TRUTHS")
print("-" * 45)

display(referential_summary)

print(
    "\nUnmatched percentage:",
    round(
        unmatched_relationships
        / len(df_hashtags_clean) * 100,
        2
    ),
    "%"
)

print("\nExamples of unmatched relationships:")

display(
    df_hashtags_clean.loc[
        ~truth_exists,
        ["id_hashtag", "hashtag", "truth_id"]
    ].head(10)
)

REFERENTIAL INTEGRITY — MAIN TRUTHS
---------------------------------------------


,Check,Count
0,Total hashtag-post relationships,205212
1,Matched relationships,168587
2,Unmatched relationships,36625
3,Distinct unmatched post IDs,11178



Unmatched percentage: 17.85 %

Examples of unmatched relationships:


,id_hashtag,hashtag,truth_id
19,5,Alaska,562384
31,5,Alaska,892604
33,5,Alaska,970375
61,7,CrookedHillary,903228
62,7,CrookedHillary,958197
77,8,SundayFunday,875879
78,8,SundayFunday,925819
79,8,SundayFunday,976192
87,10,HillaryClinton,106536
88,10,HillaryClinton,114739


In [9]:
from pathlib import Path
import pandas as pd

# Inspect available cleaned/integrated datasets
DATA_ROOT = DATA_DIR.parent

candidate_files = sorted(
    DATA_ROOT.rglob("*truth*.csv")
)

print("AVAILABLE TRUTHS DATASETS")
print("-" * 45)

for path in candidate_files:
    print(path.relative_to(DATA_ROOT))

# Inspect the main cleaned dataset schema
main_columns = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    nrows=5
)

print("\nMAIN TRUTHS COLUMNS")
print(main_columns.columns.tolist())

print("\nMAIN TRUTHS IDENTIFIERS")
display(
    main_columns[
        ["id", "external_id", "url"]
    ]
)

# Compare identifier ranges
print("\nUNMATCHED HASHTAG POST IDs")
print("Minimum:",
      df_hashtags_clean.loc[
          ~truth_exists, "truth_id"
      ].min())

print("Maximum:",
      df_hashtags_clean.loc[
          ~truth_exists, "truth_id"
      ].max())

print("Distinct unmatched IDs:",
      df_hashtags_clean.loc[
          ~truth_exists, "truth_id"
      ].nunique())

AVAILABLE TRUTHS DATASETS
---------------------------------------------
Project_Data\truths.csv
Project_Data\truths_clean.csv

MAIN TRUTHS COLUMNS
['id', 'timestamp', 'time_scraped', 'is_retruth', 'is_reply', 'author', 'like_count', 'retruth_count', 'reply_count', 'text', 'external_id', 'url', 'truth_retruthed']

MAIN TRUTHS IDENTIFIERS


,id,external_id,url
0,703265,108351978111014587,https://truthsocial.com/@jweimer39/posts/10835...
1,807614,108040697775316106,https://truthsocial.com/@CeceBloomwood/posts/1...
2,807615,108040478595657472,https://truthsocial.com/@CeceBloomwood/posts/1...
3,807618,108040420546204424,https://truthsocial.com/@CeceBloomwood/posts/1...
4,807619,108040224733670356,https://truthsocial.com/@CeceBloomwood/posts/1...



UNMATCHED HASHTAG POST IDs
Minimum: 112
Maximum: 978140
Distinct unmatched IDs: 11178


In [10]:
# Load original Truths identifiers
TRUTHS_RAW_PATH = DATA_DIR / "truths.csv"

df_truths_raw_ids = pd.read_csv(
    TRUTHS_RAW_PATH,
    usecols=["id"],
    dtype={"id": "Int64"}
)

# Identify unmatched hashtag post IDs
missing_post_ids = pd.Index(
    df_hashtags_clean.loc[
        ~truth_exists,
        "truth_id"
    ].dropna().unique()
)

# Identify original Truths identifiers
original_truth_ids = pd.Index(
    df_truths_raw_ids["id"].dropna().unique()
)

# Find posts present in original but absent from cleaned Truths
recovered_from_original = missing_post_ids.intersection(
    original_truth_ids
)

# Count affected relationships
recoverable_relationships = (
    df_hashtags_clean["truth_id"]
    .isin(recovered_from_original)
    .sum()
)

print("ORIGINAL TRUTHS RECOVERY ASSESSMENT")
print("-" * 45)

print("Distinct unmatched post IDs:",
      len(missing_post_ids))

print("Post IDs found in original Truths:",
      len(recovered_from_original))

print("Potentially recoverable relationships:",
      recoverable_relationships)

print("\nExamples of matching IDs:")
print(recovered_from_original[:20].tolist())

ORIGINAL TRUTHS RECOVERY ASSESSMENT
---------------------------------------------
Distinct unmatched post IDs: 11178
Post IDs found in original Truths: 0
Potentially recoverable relationships: 0

Examples of matching IDs:
[]


In [11]:
# Identify hashtags associated with unmatched posts
df_unmatched_hashtags = df_hashtags_clean.loc[
    ~truth_exists,
    ["truth_id", "hashtag"]
].copy()

print("CONTENT-BASED RECOVERY ASSESSMENT")
print("-" * 45)

print("Unmatched hashtag relationships:",
      len(df_unmatched_hashtags))

print("Distinct unmatched posts:",
      df_unmatched_hashtags["truth_id"].nunique())

print("Distinct hashtag names:",
      df_unmatched_hashtags["hashtag"].nunique())

# Check whether additional Truths is available
if "df_additional" in globals():

    print("\nAdditional Truths DataFrame available.")

    print("Shape:", df_additional.shape)

    print("\nAvailable columns:")
    print(df_additional.columns.tolist())

    print("\nSample records:")
    display(df_additional.head(3))

else:

    print(
        "\nAdditional Truths DataFrame not loaded "
        "in this notebook."
    )

CONTENT-BASED RECOVERY ASSESSMENT
---------------------------------------------
Unmatched hashtag relationships: 36625
Distinct unmatched posts: 11178
Distinct hashtag names: 4903

Additional Truths DataFrame not loaded in this notebook.


In [12]:
from pathlib import Path
import importlib.util

# Define ARFF file path
ADDITIONAL_PATH = (
    DATA_DIR.parent
    / "Additional_Data"
    / "additional_truths.arff"
)

if not ADDITIONAL_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {ADDITIONAL_PATH}"
    )

print("ADDITIONAL TRUTHS FILE")
print("-" * 40)

print("Path:", ADDITIONAL_PATH)

print("File size (MB):",
      round(ADDITIONAL_PATH.stat().st_size / 1024**2, 2))

print("\nAvailable parsing libraries:")

for module in ["arff", "scipy", "pandas"]:
    print(
        module,
        importlib.util.find_spec(module) is not None
    )

ADDITIONAL TRUTHS FILE
----------------------------------------
Path: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Additional_Data\additional_truths.arff
File size (MB): 13.13

Available parsing libraries:
arff True
scipy True
pandas True


In [14]:
import json
from pathlib import Path

# Actual notebook filename
NOTEBOOK_NAME = "01_12_clean_integr_addtruths.ipynb"

# Search in the project directory and its parents
search_roots = [
    Path.cwd(),
    *Path.cwd().parents,
    DATA_DIR.parent,
    DATA_DIR.parent.parent
]

notebooks = []

for root in search_roots:
    if root.exists():
        notebooks.extend(
            root.glob(f"01_assignment/{NOTEBOOK_NAME}")
        )

# Remove duplicate paths
notebooks = list(dict.fromkeys(notebooks))

if not notebooks:
    raise FileNotFoundError(
        f"Notebook not found: {NOTEBOOK_NAME}"
    )

notebook_path = notebooks[0]

print("Notebook found:", notebook_path)

# Read notebook
with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

# Search for ARFF loading code
keywords = [
    "arff.load",
    "loadarff",
    "csv.reader",
    "read_csv",
    "@data",
    "df_additional",
]

print("\nARFF LOADING CELLS")
print("-" * 50)

for i, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    if any(keyword in source for keyword in keywords):
        print(f"\n--- CELL {i} ---")
        print(source[:5000])

Notebook found: c:\Users\bucco\Desktop\Laboratory of Data Science\project\Truth-Social-Analysis-and-Data-Warehousing---Decision-Support-Systems-\01_assignment\01_12_clean_integr_addtruths.ipynb

ARFF LOADING CELLS
--------------------------------------------------

--- CELL 0 ---

from pathlib import Path
import pandas as pd

BASE_PATH = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data"
)

TRUTHS_CLEAN_PATH = (
    BASE_PATH / "Project_Data" / "truths_clean.csv"
)

ADDITIONAL_PATH = (
    BASE_PATH / "Additional_Data" / "additional_truths.arff"
)

# Verify input files
for path in [TRUTHS_CLEAN_PATH, ADDITIONAL_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

# Load the cleaned main corpus
# Preserve external IDs as strings to avoid precision loss
df_truths = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    dtype={
        "external_id": "string",
        "url": "string"
    },
    low_memory=False
)

pri

In [15]:
# Identify the ARFF parsing procedure
parsing_keywords = [
    "csv.reader",
    "arff.load",
    "loadarff",
    "@data",
    "def parse",
    "def load",
    "split(',',",
    'split(",",',
]

print("ARFF PARSING PROCEDURE")
print("-" * 45)

for i, cell in enumerate(notebook["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    # Skip unrelated cells
    if not any(
        keyword in source
        for keyword in parsing_keywords
    ):
        continue

    print(f"\n--- CELL {i} ---")
    print(source[:3500])

ARFF PARSING PROCEDURE
---------------------------------------------

--- CELL 1 ---

# Load additional_truths.arff using the validated parsing strategy

additional_columns = [
    "id",
    "timestamp",
    "text",
    "url",
    "reply_count",
    "retruth_count",
    "like_count",
    "external_id",
    "author",
    "time_scraped"
]

records = []
data_started = False

with open(ADDITIONAL_PATH, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        if line.lower() == "@data":
            data_started = True
            continue

        if not data_started or not line or line.startswith("%"):
            continue

        # First two attributes from the left
        left = line.split(",", 2)

        # Remaining seven attributes from the right
        right = left[2].rsplit(",", 7)

        row = [left[0], left[1]] + right

        if len(row) != len(additional_columns):
            raise ValueError(
                f"Invalid ARFF record with {le

In [19]:

# Load additional_truths.arff using the validated parsing strategy

additional_columns = [
    "id",
    "timestamp",
    "text",
    "url",
    "reply_count",
    "retruth_count",
    "like_count",
    "external_id",
    "author",
    "time_scraped"
]

records = []
data_started = False

with open(ADDITIONAL_PATH, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()

        if line.lower() == "@data":
            data_started = True
            continue

        if not data_started or not line or line.startswith("%"):
            continue

        # First two attributes from the left
        left = line.split(",", 2)

        # Remaining seven attributes from the right
        right = left[2].rsplit(",", 7)

        row = [left[0], left[1]] + right

        if len(row) != len(additional_columns):
            raise ValueError(
                f"Invalid ARFF record with {len(row)} fields"
            )

        records.append(row)

df_additional = pd.DataFrame(
    records,
    columns=additional_columns
)

# Remove enclosing quotes without changing internal text
def remove_enclosing_quotes(value):
    value = value.strip()

    if len(value) >= 2 and value[0] == value[-1]:
        if value[0] in ("'", '"'):
            return value[1:-1]

    return value

for col in df_additional.columns:
    df_additional[col] = (
        df_additional[col].map(remove_enclosing_quotes)
    )

# Normalize identifiers without converting large IDs to floats
for col in ["id", "external_id"]:
    df_additional[col] = (
        df_additional[col].astype("string").str.strip()
    )

print("ADDITIONAL TRUTHS LOADING")
print("-" * 45)

print("Shape:", df_additional.shape)

print("\nIDENTIFIER VALIDATION")

id_match_mask = (
    df_additional["id"]
    .eq(df_additional["external_id"])
    .fillna(False)
)

print(
    "ID / external_id matches:",
    f"{id_match_mask.sum():,}"
)

print(
    "ID / external_id mismatches:",
    f"{(~id_match_mask).sum():,}"
)

print(
    "Duplicated external IDs:",
    f"{df_additional['external_id'].duplicated().sum():,}"
)

print("\nSAMPLE RECORDS")
display(df_additional.head())


ADDITIONAL TRUTHS LOADING
---------------------------------------------
Shape: (35222, 10)

IDENTIFIER VALIDATION
ID / external_id matches: 35,222
ID / external_id mismatches: 0
Duplicated external IDs: 0

SAMPLE RECORDS


,id,timestamp,text,url,reply_count,retruth_count,like_count,external_id,author,time_scraped
0,117323511508375196,2026-09-24 01:51:54,,https://truthsocial.com/@realDonaldTrump/11732...,792.0,1849.0,7259.0,117323511508375196,27.0,2026-09-20 14:00:00.000000
1,117323155721195611,2026-09-24 00:21:25,"The story that the White House, or me, want to...",https://truthsocial.com/@realDonaldTrump/11732...,1329.0,3534.0,12560.0,117323155721195611,27.0,2026-09-20 14:00:00.000000
2,117323148147220269,2026-09-24 00:19:29,BIG DAY FOR BOEING AND AMERICAN MANUFACTURING!...,https://truthsocial.com/@realDonaldTrump/11732...,510.0,2714.0,10677.0,117323148147220269,27.0,2026-09-20 14:00:00.000000
3,117322092055919832,2026-09-23 19:50:55,,https://truthsocial.com/@realDonaldTrump/11732...,1344.0,2740.0,11287.0,117322092055919832,27.0,2026-09-20 14:00:00.000000
4,117322065119445871,2026-09-23 19:44:04,"The Great Gary Player, Bryson DeChambeau, Mika...",https://truthsocial.com/@realDonaldTrump/11732...,505.0,2156.0,10284.0,117322065119445871,27.0,2026-09-20 14:00:00.000000


In [20]:
# Validate the loaded Additional Truths dataset
print("ADDITIONAL TRUTHS VALIDATION")
print("-" * 45)

print("Dataset shape:", df_additional.shape)

print("\nAvailable columns:")
print(df_additional.columns.tolist())

print("\nMissing post texts:")
print(df_additional["text"].isna().sum())

print("\nDuplicate post IDs:")
print(df_additional["id"].duplicated().sum())

print("\nSample records:")
display(
    df_additional[
        ["id", "text", "url"]
    ].head(3)
)

ADDITIONAL TRUTHS VALIDATION
---------------------------------------------
Dataset shape: (35222, 10)

Available columns:
['id', 'timestamp', 'text', 'url', 'reply_count', 'retruth_count', 'like_count', 'external_id', 'author', 'time_scraped']

Missing post texts:
0

Duplicate post IDs:
0

Sample records:


,id,text,url
0,117323511508375196,,https://truthsocial.com/@realDonaldTrump/11732...
1,117323155721195611,"The story that the White House, or me, want to...",https://truthsocial.com/@realDonaldTrump/11732...
2,117323148147220269,BIG DAY FOR BOEING AND AMERICAN MANUFACTURING!...,https://truthsocial.com/@realDonaldTrump/11732...


In [21]:
import re
import pandas as pd

# Identify hashtags associated with unmatched posts
df_unmatched_hashtags = df_hashtags_clean.loc[
    ~truth_exists,
    ["truth_id", "hashtag"]
].copy()

# Build a normalized set of candidate hashtags
candidate_hashtags = set(
    df_unmatched_hashtags["hashtag"]
    .dropna()
    .astype(str)
    .str.casefold()
)

# Extract hashtags from additional post texts
hashtag_pattern = re.compile(r"(?<!\w)#([\w]+)", re.UNICODE)

def extract_hashtags(text):
    if pd.isna(text):
        return set()

    return {
        tag.casefold()
        for tag in hashtag_pattern.findall(str(text))
    }

df_additional_hashtag_check = df_additional[
    ["id", "text"]
].copy()

df_additional_hashtag_check["extracted_hashtags"] = (
    df_additional_hashtag_check["text"]
    .apply(extract_hashtags)
)

# Identify overlap with unmatched hashtags
df_additional_hashtag_check["matching_hashtags"] = (
    df_additional_hashtag_check["extracted_hashtags"]
    .apply(lambda tags: tags & candidate_hashtags)
)

df_additional_hashtag_check["has_match"] = (
    df_additional_hashtag_check["matching_hashtags"]
    .apply(bool)
)

# Summarize results
matching_posts = df_additional_hashtag_check[
    df_additional_hashtag_check["has_match"]
]

matched_hashtag_names = set().union(
    *matching_posts["matching_hashtags"].tolist()
) if len(matching_posts) > 0 else set()

print("HASHTAG CONTENT OVERLAP")
print("-" * 45)

print("Distinct unmatched hashtag names:",
      len(candidate_hashtags))

print("Additional Truths posts:",
      len(df_additional_hashtag_check))

print("Additional posts with matching hashtags:",
      len(matching_posts))

print("Distinct matching hashtag names:",
      len(matched_hashtag_names))

print("\nExamples of matching posts:")

display(
    matching_posts[
        ["id", "matching_hashtags"]
    ].head(20)
)

HASHTAG CONTENT OVERLAP
---------------------------------------------
Distinct unmatched hashtag names: 4208
Additional Truths posts: 35222
Additional posts with matching hashtags: 311
Distinct matching hashtag names: 169

Examples of matching posts:


,id,matching_hashtags
1980,116868292534134605,{trump}
2859,116655731323552591,{maga}
6486,115565538219866607,{xteam}
9727,114504178725559243,{truth}
9728,114504177179345854,"{maga, americafirst}"
9731,114504169401516645,"{truth, maga, americafirst}"
9735,114504162128518905,"{truth, maga, americafirst}"
9736,114504159947585102,"{maga, truth}"
9794,114484272669802796,{metoo}
11418,113984864129540464,"{americafirst, saveamerica, truth}"


In [22]:
# Final Data Quality Assessment

id_columns = ["id_hashtag", "hashtag_id", "truth_id"]

# Check missing values
missing_values = df_hashtags_clean.isna().sum().sum()

# Check invalid identifiers
invalid_ids = (
    df_hashtags_clean[id_columns]
    .le(0)
    .sum()
    .sum()
)

# Check duplicate records and relationships
duplicate_records = df_hashtags_clean.duplicated().sum()

duplicate_pairs = df_hashtags_clean.duplicated(
    subset=["id_hashtag", "truth_id"]
).sum()

# Check identifier consistency
id_mismatches = (
    df_hashtags_clean["id_hashtag"]
    != df_hashtags_clean["hashtag_id"]
).fillna(False).sum()

# Check hashtag mapping consistency
inconsistent_names = (
    df_hashtags_clean.groupby("id_hashtag")["hashtag"]
    .nunique()
    .gt(1)
    .sum()
)

# Check timestamps
invalid_timestamps = (
    df_hashtags_clean["first_seen"].isna().sum()
)

# Check remaining malformed hashtag values
malformed_values = (
    df_hashtags_clean["hashtag"]
    .astype("string")
    .str.contains(
        r'rel\s*=\s*["\']|<[^>]+>|\s',
        regex=True,
        na=False
    )
    .sum()
)

# Recalculate referential integrity
unmatched_references = (
    ~df_hashtags_clean["truth_id"].isin(valid_truth_ids)
).sum()

# Build quality report
quality_report = pd.DataFrame({
    "Check": [
        "Missing values",
        "Invalid identifiers",
        "Duplicate records",
        "Duplicate hashtag-post relationships",
        "Hashtag ID mismatches",
        "IDs associated with multiple names",
        "Invalid timestamps",
        "Malformed hashtag values",
        "Unmatched post references"
    ],
    "Count": [
        missing_values,
        invalid_ids,
        duplicate_records,
        duplicate_pairs,
        id_mismatches,
        inconsistent_names,
        invalid_timestamps,
        malformed_values,
        unmatched_references
    ]
})

print("FINAL DATA QUALITY ASSESSMENT")
print("-" * 45)

display(quality_report)

print("\nDATASET COMPARISON")
print("-" * 45)

print("Original records:", len(df_hashtags_raw))
print("Duplicates removed:", exact_duplicates)
print("Final records:", len(df_hashtags_clean))

# Validate structural integrity
assert missing_values == 0
assert invalid_ids == 0
assert duplicate_records == 0
assert duplicate_pairs == 0
assert id_mismatches == 0
assert inconsistent_names == 0
assert invalid_timestamps == 0
assert malformed_values == 0

assert len(df_hashtags_clean) == (
    len(df_hashtags_raw) - exact_duplicates
)

print("\nAll internal data quality checks passed.")

FINAL DATA QUALITY ASSESSMENT
---------------------------------------------


,Check,Count
0,Missing values,0
1,Invalid identifiers,0
2,Duplicate records,0
3,Duplicate hashtag-post relationships,0
4,Hashtag ID mismatches,0
5,IDs associated with multiple names,0
6,Invalid timestamps,0
7,Malformed hashtag values,0
8,Unmatched post references,36625



DATASET COMPARISON
---------------------------------------------
Original records: 205214
Duplicates removed: 2
Final records: 205212

All internal data quality checks passed.


In [23]:
# Preserve original schema
df_hashtags_final = df_hashtags_clean[
    [
        "id_hashtag",
        "first_seen",
        "hashtag",
        "truth_id",
        "hashtag_id"
    ]
].copy()

# Define export path
HASHTAGS_CLEAN_PATH = DATA_DIR / "hashtags_clean.csv"

# Export cleaned dataset
df_hashtags_final.to_csv(
    HASHTAGS_CLEAN_PATH,
    index=False
)

# Final validation
assert df_hashtags_final.shape == (205212, 5)

print("EXPORT COMPLETED")
print("-" * 40)

print("File:", HASHTAGS_CLEAN_PATH)

print("\nFinal dataset shape:")
print(df_hashtags_final.shape)

print("\nFinal dataset preview:")
display(df_hashtags_final.head())

EXPORT COMPLETED
----------------------------------------
File: C:\Users\bucco\Desktop\Laboratory of Data Science\project\code_project\data\Project_Data\hashtags_clean.csv

Final dataset shape:
(205212, 5)

Final dataset preview:


,id_hashtag,first_seen,hashtag,truth_id,hashtag_id
0,1,2022-09-14 17:29:14.578577+00:00,MarkGeist,18,1
1,2,2022-09-14 17:34:04.180279+00:00,LIVGolf,21,2
2,2,2022-09-14 17:34:04.180279+00:00,LIVGolf,478624,2
3,3,2022-09-14 23:15:11.267669+00:00,TrumpTurnberry,37,3
4,3,2022-09-14 23:15:11.267669+00:00,TrumpTurnberry,384189,3
